# Decision Tree

In [26]:
import math


In [27]:
# Example dataset
# let's say we have a dataset of two class A and B
# Number of elements in each class
class_A = 4
class_B = 6
# Total number of elements
total = class_A + class_B

# Proportional 

In [28]:
# proportion
P_a = class_A / total
p_b = class_B / total
# print the proportion
print(P_a)
print(p_b)

0.4
0.6


In [29]:
# Now we calculate the entropy
entropy = - (P_a * math.log2(P_a) + p_b * math.log2(p_b))
print(entropy)

0.9709505944546686


In [30]:
# now gini impurity
gini_impurity = 1 - (P_a ** 2 + p_b ** 2)
print(gini_impurity)

0.48


In [31]:
# information gain
# let's say we have a feature that splits the dataset into two subsets
# subset 1 has 3 elements of class A and 2 elements of class B  
# subset 2 has 1 element of class A and 4 elements of class B
subset_1_A = 3
subset_1_B = 2
subset_2_A = 1
subset_2_B = 4
# total elements in each subset
total_subset_1 = subset_1_A + subset_1_B
total_subset_2 = subset_2_A + subset_2_B
# proportion in each subset
P_a_subset_1 = subset_1_A / total_subset_1
p_b_subset_1 = subset_1_B / total_subset_1
P_a_subset_2 = subset_2_A / total_subset_2
p_b_subset_2 = subset_2_B / total_subset_2
# entropy of each subset
entropy_subset_1 = - (P_a_subset_1 * math.log2(P_a_subset_1) + p_b_subset_1 * math.log2(p_b_subset_1))
entropy_subset_2 = - (P_a_subset_2 * math.log2(P_a_subset_2) + p_b_subset_2 * math.log2(p_b_subset_2))
# weighted average entropy of the subsets
weighted_entropy = (total_subset_1 / total) * entropy_subset_1 + (total_subset_2 / total) * entropy_subset_2
# information gain  
information_gain = entropy - weighted_entropy
print(information_gain)


0.12451124978365313


----


# Decision Tree

In [32]:
# Import libraries 
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer


In [33]:
# load the titanic dataset
df = sns.load_dataset('titanic')
df.head()



,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


In [34]:
df.isnull().sum()

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

In [35]:
# drop the deck column 
df.drop('deck', axis=1, inplace=True)
# impute the missing values in the age and fare column with the mean
imputer = SimpleImputer(strategy='mean')
df[['age', 'fare']] = imputer.fit_transform(df[['age', 'fare']])
# impute the missing values of embark and embark_town with the mode
imputer = SimpleImputer(strategy='most_frequent')
df[['embark_town', 'embarked']] = imputer.fit_transform(df[['embark_town', 'embarked']])



In [36]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True


In [37]:
# encode the categorical and objects variables using for loop
le = LabelEncoder()
for column in df.columns:
    if df[column].dtype == 'object' or df[column].dtype.name == 'category':
        df[column] = le.fit_transform(df[column])

In [38]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    int32  
 3   age          891 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     891 non-null    int32  
 8   class        891 non-null    int32  
 9   who          891 non-null    int32  
 10  adult_male   891 non-null    bool   
 11  embark_town  891 non-null    int32  
 12  alive        891 non-null    int32  
 13  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int32(6), int64(4)
memory usage: 64.5 KB


In [39]:
df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone
0,0,3,1,22.0,1,0,7.2500,2,2,1,True,2,0,False
1,1,1,0,38.0,1,0,71.2833,0,0,2,False,0,1,False
2,1,3,0,26.0,0,0,7.9250,2,2,2,False,2,1,True
3,1,1,0,35.0,1,0,53.1000,2,0,2,False,2,1,False
4,0,3,1,35.0,0,0,8.0500,2,2,1,True,2,0,True


In [40]:
# split the data into x and y
X = df.drop('survived', axis=1)
y = df['survived']

In [41]:
# split the data into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [42]:
# create and trian the model 
model = DecisionTreeClassifier(criterion='entropy', max_depth=3, random_state=42)
model.fit(X_train, y_train)
# predict the model
y_pred = model.predict(X_test)
# evaluate the model 
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))



Accuracy: 1.0
Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       105
           1       1.00      1.00      1.00        74

    accuracy                           1.00       179
   macro avg       1.00      1.00      1.00       179
weighted avg       1.00      1.00      1.00       179



In [47]:
# now save the decision classifier

from sklearn.tree import export_graphviz
# export the decision tree to a dot file
export_graphviz(model, out_file='decision_tree.dot', feature_names=X.columns, class_names=['Not Survived', 'Survived'], filled=True)